## Week 2 Day 1

And now! Our first look at OpenAI Agents SDK

You won't believe how lightweight this is..

<table style="margin: 0; text-align: left; width:100%">
    <tr>
        <td style="width: 150px; height: 150px; vertical-align: middle;">
            <img src="../assets/tools.png" width="150" height="150" style="display: block;" />
        </td>
        <td>
            <h2 style="color:#00bfff;">The OpenAI Agents SDK Docs</h2>
            <span style="color:#00bfff;">The documentation on OpenAI Agents SDK is really clear and simple: <a href="https://openai.github.io/openai-agents-python/">https://openai.github.io/openai-agents-python/</a> and it's well worth a look.
            </span>
        </td>
    </tr>
</table>

# Three Parts to this lab

## Part 1: A simple "Agent" and "Agent Loop"

Basically an LLM call. We'll add tracing and streaming to the mix.

## Part 2: Adding a Tool

A familiar one, but oh-so-easy

## Part 3: Adding Memory

So that different Agent calls know about each other

In [13]:
# The imports

import os
import requests
from dotenv import load_dotenv
from openai.types.responses import ResponseTextDeltaEvent
from agents import Agent, Runner, trace, function_tool, SQLiteSession
load_dotenv(override=True)


True

## Sidenote

The actual name of this framework on the official Python index pypi.org is `openai-agents`

So for your own projects in the future, you would do:

`pip install openai-agents`  
or  
`uv add openai-agents`

followed by

`from agents import Agent, Runner, trace`

Beware that doing a `pip install agents` would install something completely different - an older reinforcement learning library.


In [14]:
# The imports

from dotenv import load_dotenv
from agents import Agent, Runner, trace



In [15]:
# The usual starting point

load_dotenv(override=True)


True

In [16]:

# Make an agent with name, instructions, model
# You have to give the agent a name. instructions are basically systemp prompt. 
# for the model, if it's a string, it assumes that it's an OpenAI model. If you want to use Gemini, you will need to add a few more lines of code with different guidelines.

agent = Agent(name="Jokester", instructions="You are a joke teller", model="gpt-4o-mini")

In [17]:
# If you only do Runner.run, you will see that the output is just a coroutine object. Runner.run does not actually run the agent.
Runner.run(agent, "Tell a joke about Autonomous AI Agents") 

<coroutine object Runner.run at 0x117b093f0>

In [18]:
# If you add "await" in front of Runner.run, it will schedule the coroutine to run. 
result = await Runner.run(agent, "Tell a joke about Autonomous AI Agents") 
result

RunResult(input='Tell a joke about Autonomous AI Agents', new_items=[MessageOutputItem(agent=Agent(name='Jokester', handoff_description=None, tools=[], mcp_servers=[], mcp_config={}, instructions='You are a joke teller', prompt=None, handoffs=[], model='gpt-4o-mini', model_settings=ModelSettings(temperature=None, top_p=None, frequency_penalty=None, presence_penalty=None, tool_choice=None, parallel_tool_calls=None, truncation=None, max_tokens=None, reasoning=None, verbosity=None, metadata=None, store=None, include_usage=None, response_include=None, top_logprobs=None, extra_query=None, extra_body=None, extra_headers=None, extra_args=None), input_guardrails=[], output_guardrails=[], output_type=None, hooks=None, tool_use_behavior='run_llm_again', reset_tool_choice=True), raw_item=ResponseOutputMessage(id='msg_0493c17a0a4d5b9f006a4320397c4881998a21a52b5a201cbf', content=[ResponseOutputText(annotations=[], text='Why did the autonomous AI agent break up with its partner?\n\nBecause it needed

In [19]:
result.to_input_list()

[{'content': 'Tell a joke about Autonomous AI Agents', 'role': 'user'},
 {'id': 'msg_0493c17a0a4d5b9f006a4320397c4881998a21a52b5a201cbf',
  'content': [{'annotations': [],
    'text': 'Why did the autonomous AI agent break up with its partner?\n\nBecause it needed some "space" to process its feelings!',
    'type': 'output_text',
    'logprobs': []}],
  'role': 'assistant',
  'status': 'completed',
  'type': 'message'}]

## Adding Observability with a trace

In [20]:
# Run the joke with Runner.run(agent, prompt) then print final_output

with trace("Telling a joke"): # You can just use trace(), but give it a specific identification is helpful. 
    result = await Runner.run(agent, "Tell a joke about Autonomous AI Agents") 
    print(result.final_output)  # Access the "final_output" to get the actual output of the agent.

Why did the autonomous AI agent break up with its partner?

Because it found someone who truly understood its algorithms!


In [21]:
# with trace() adds observability to the agent. Tracing is built into the OpenAI SDK. It's part of the free open source software from OpenAI. It's not directly related to the actual call of the model. It's not the LLM tracing - it's the agent framework. So whatever model you use, the model running will still get traced. the only think you need is an OpenAI API Key. But you won't need any money or balance at all. 

## Now go and look at the trace

https://platform.openai.com/traces

On the OpenAI Platform web, you will see all the traces. If you gave the trace() a name, it will show that name. Otherwise, it will just show the default "Agent workflow."
For each trace, you will see all the timeline, the user input, assistant output, etc. 

Note: on OpenAI platform web, go to the project that you use the OpenAI API key for the agent to see the traces. 

In [22]:
# Streaming: instead of getting the final output, you can get the output as it's being generated. This is how you get "streaming" in the UI. 

from openai.types.responses import ResponseTextDeltaEvent


result = Runner.run_streamed(agent, input="Tell three jokes about Autonomous AI Agents")
async for event in result.stream_events():
    if event.type == "raw_response_event" and isinstance(event.data, ResponseTextDeltaEvent):
        print(event.data.delta, end="", flush=True)


Sure! Here are three jokes about Autonomous AI Agents:

1. **Why did the Autonomous AI Agent break up with its partner?**  
   Because it couldn't handle all the emotional variables!

2. **How do Autonomous AI Agents stay organized?**  
   They use cloud storage—too many “thoughts” in one place can lead to a system crash!

3. **Why did the Autonomous AI Agent get promoted?**  
   Because it always went above and beyond its programming—and didn't even need coffee breaks!

## Part 2: Adding a tool

In [23]:
pushover_user = os.getenv("PUSHOVER_USER")
pushover_token = os.getenv("PUSHOVER_TOKEN")
pushover_url = "https://api.pushover.net/1/messages.json"

if pushover_user:
    if pushover_user.startswith("u"):
        print("Pushover user found and looks good")
    else:
        print("Pushover user found but doesn't start with u")
else:
    print("Pushover user not found")

if pushover_token:
    if pushover_token.startswith("a"):
        print("Pushover token found and looks good")
    else:
        print("Pushover token found but doesn't start with a")
else:
    print("Pushover token not found")

Pushover user found and looks good
Pushover token found and looks good


In [24]:
# Remember this?

def push(message):
    print(f"Push: {message}")
    payload = {"user": pushover_user, "token": pushover_token, "message": message}
    requests.post(pushover_url, data=payload)

In [25]:
push("HEY!!")

Push: HEY!!


In [26]:
push

<function __main__.push(message)>

In [27]:
# Now this:

@function_tool
def push_tool(message: str) -> str:
    """ Send the given message to the user as a push notification """
    payload = {"user": pushover_user, "token": pushover_token, "message": message}
    result = requests.post(pushover_url, data=payload).status_code
    return f"Push sent with API status code {result}"

In [28]:
push_tool

FunctionTool(name='push_tool', description='Send the given message to the user as a push notification', params_json_schema={'properties': {'message': {'title': 'Message', 'type': 'string'}}, 'required': ['message'], 'title': 'push_tool_args', 'type': 'object', 'additionalProperties': False}, on_invoke_tool=<function function_tool.<locals>._create_function_tool.<locals>._on_invoke_tool at 0x1166c0720>, strict_json_schema=True, is_enabled=True, tool_input_guardrails=None, tool_output_guardrails=None)

In [29]:
push_tool.description

'Send the given message to the user as a push notification'

In [30]:

notifier = Agent(name="Notifier", model="gpt-5.4-mini", instructions="You notify the user upon request", tools=[push_tool])

In [31]:
with trace("Pizza has arrived"):
    result = await Runner.run(notifier, "Notify the user that the pizza is here")

print(result.final_output)


Done.


## Now go and look at the trace

https://platform.openai.com/traces

## Part 3: Sessions (memory)

Within a Runner.run() application level turn, the conversation history is maintained.

But each call to Runner.run() is a fresh start.

Let's see that:

In [32]:
agent = Agent(name="Assistant", model="gpt-5.4-mini")

In [33]:
response = await Runner.run(agent, "Hi there. My name is Ed.")
print(response.final_output)

Hi Ed — nice to meet you. How can I help today?


In [34]:
response = await Runner.run(agent, "What's my name?")
print(response.final_output)

I don’t know your name yet. If you want, tell me what you’d like me to call you.


## Memory approach 1 - just manually pass in the list of dicts

In [35]:
response = await Runner.run(agent, "Hi there. My name is Ed.")
print(response.final_output)

Hi Ed — nice to meet you. How can I help?


In [36]:
response.to_input_list()

[{'content': 'Hi there. My name is Ed.', 'role': 'user'},
 {'id': 'msg_005b93f4363cc28c006a432043eb648198a6b9e4b03110395f',
  'content': [{'annotations': [],
    'text': 'Hi Ed — nice to meet you. How can I help?',
    'type': 'output_text',
    'logprobs': []}],
  'role': 'assistant',
  'status': 'completed',
  'type': 'message',
  'phase': 'final_answer'}]

In [37]:
next_input = response.to_input_list() + [{"role": "user", "content": "What's my name?"}]
next_input

[{'content': 'Hi there. My name is Ed.', 'role': 'user'},
 {'id': 'msg_005b93f4363cc28c006a432043eb648198a6b9e4b03110395f',
  'content': [{'annotations': [],
    'text': 'Hi Ed — nice to meet you. How can I help?',
    'type': 'output_text',
    'logprobs': []}],
  'role': 'assistant',
  'status': 'completed',
  'type': 'message',
  'phase': 'final_answer'},
 {'role': 'user', 'content': "What's my name?"}]

In [38]:
response = await Runner.run(agent, next_input)
print(response.final_output)

Your name is Ed.


## Another approach - use OpenAI Agents SDK built in SQLLite session

In [39]:
# This is created in-memory
# For an on-disk memory, use SQLiteSession("12345", "memory.db")

session = SQLiteSession("12346")

In [40]:
response = await Runner.run(agent, "Hi there. My name is Ed.", session=session)
print(response.final_output)

Hi Ed — nice to meet you! How can I help you today?


In [41]:
response = await Runner.run(agent, "What's my name?", session=session)
print(response.final_output)

Your name is Ed.


# WOW

Can you believe how much we got done in Lab 1?!

Agents, Runner (Agent Loop), traces (Observability), Streaming, Function Tools, Memory!

Remember to check out the docs:  
https://openai.github.io/openai-agents-python/

Even better news: many of the lightweight Agent Frameworks are very similar, so you practically know them all..


<table style="margin: 0; text-align: left; width:100%">
    <tr>
        <td style="width: 150px; height: 150px; vertical-align: middle;">
            <img src="../assets/exercise.png" width="150" height="150" style="display: block;" />
        </td>
        <td>
            <h2 style="color:#ff7800;">Exercise</h2>
            <span style="color:#ff7800;">Make one of the Week 1 projects using OpenAI Agents SDK - like the digital twin or the Checklist loop. You will be astonished how easy it is.
            </span>
        </td>
    </tr>
</table>